In [ ]:
from functools import partial

import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
features = pl.read_parquet("data/candidates.10k.parquet")
site1_labels = pl.read_parquet("data/1L83.1L83:p2rank:2.10k.parquet")

In [ ]:
site1_labels = site1_labels.unique(subset=["catalog_id"], keep="first")

In [ ]:
cns_mpo_schema = pl.Struct(
    [
        pl.Field("clogp", pl.Float64),
        pl.Field("clogd", pl.Float64),
        pl.Field("tpsa", pl.Float64),
    ]
)

In [ ]:
features = features.with_columns(
    pl.col("cns_mpo_components").str.json_decode(cns_mpo_schema)
).unnest("cns_mpo_components")

In [ ]:
base_fields = [
    pl.Field("radius_of_gyration", pl.Float64),
    pl.Field("pmi1_normalized", pl.Float64),
    pl.Field("pmi2_normalized", pl.Float64),
    pl.Field("asphericity", pl.Float64),
    pl.Field("eccentricity", pl.Float64),
    pl.Field("spherocity_index", pl.Float64),
]

autocorr_fields = [pl.Field(f"autocorr3d_{i}", pl.Float64) for i in range(80)]


conformer_schema = pl.Struct([*base_fields, *autocorr_fields])

descriptors_field_names = [field.name for field in conformer_schema.fields]

In [ ]:
features = features.with_columns(
    pl.col("descriptors_json").str.json_decode(conformer_schema)
).unnest("descriptors_json")

In [ ]:
features = features.drop(["smiles", "cns_mpo", "parse_ok", "pains_flags"])

In [ ]:
site1_labels = site1_labels.drop(
    [
        "conformational_state_id",
        "site_id",
        "conformational_state_id",
        "pose_pdbqt",
        "rmsd_lb",
        "rmsd_ub",
    ]
)

In [ ]:
site1_df = features.join(site1_labels, on="catalog_id", how="inner")

In [ ]:
site1_y = site1_df["affinity_kcal_mol"].to_numpy()

In [ ]:
numerical_cols = [
    "heavy_atom_count",
    "molecular_weight",
    "clogp",
    "clogd",
    "tpsa",
    *descriptors_field_names,
]
site1_X_num = site1_df.select(numerical_cols).to_numpy()

In [ ]:
site1_X = np.hstack([site1_X_num, np.array(site1_df["morgan_fp"].to_list())])

In [ ]:
site1_X_train, site1_X_test, site1_y_train, site1_y_test = train_test_split(
    site1_X, site1_y, test_size=0.2, random_state=42
)

In [ ]:
def objective(trial, X_train, y_train):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "random_state": 42,
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "verbose": -1,
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []

    for train_idx, val_idx in kf.split(X_train, y_train):
        X_tr, y_tr = X_train[train_idx], y_train[train_idx]
        X_va, y_va = X_train[val_idx], y_train[val_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_X=X_va,
            eval_y=y_va,
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
        )

        preds = model.predict(X_va)
        rmse = np.sqrt(mean_squared_error(y_va, preds))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

In [ ]:
site1_study = optuna.create_study(direction="minimize")

In [ ]:
site1_objective = partial(objective, X_train=site1_X_train, y_train=site1_y_train)

site1_study.optimize(site1_objective, n_trials=30, show_progress_bar=True)

In [ ]:
print(f"\nBest Cross-Validation RMSE: {site1_study.best_value:.4f} kcal/mol")
print("Best Hyperparameters:")
for k, v in site1_study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
site1_best_params = site1_study.best_params
site1_best_params.update({"n_estimators": 1000, "random_state": 42, "verbose": -1})

site1_final_model = lgb.LGBMRegressor(**site1_best_params)
site1_final_model.fit(
    site1_X_train,
    site1_y_train,
    eval_X=site1_X_test,
    eval_y=site1_y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

In [ ]:
site1_y_pred = site1_final_model.predict(site1_X_test)
site1_test_rmse = np.sqrt(mean_squared_error(site1_y_test, site1_y_pred))
site1_test_r2 = r2_score(site1_y_test, site1_y_pred)

print("--- Final Model Test Results ---")
print(f"Tuned Test RMSE: {site1_test_rmse:.4f} kcal/mol")
print(f"Tuned Test R²:   {site1_test_r2:.4f}")